In [19]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import copy

from sklearn.linear_model import LogisticRegression,ElasticNet
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
# 评价方法
from sklearn.metrics import roc_curve, roc_auc_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split,StratifiedKFold

import lightgbm as lgb
#from catboost import CatBoostClassifier
from tqdm import tqdm
import time 
import datetime

In [20]:
fea_df = pd.read_pickle('fea_df2.pkl')

In [21]:
data=fea_df

In [22]:
#candle_begin_time,symbol,is_spot,close
TIME='candle_begin_time'
TARGET='Return_24'
USE_COLS = data.columns.difference(['candle_begin_time','symbol','is_spot','close','Return_24'])

In [23]:
data['count'] = data.groupby('candle_begin_time')['symbol'].transform('count') - 1

In [24]:
data[TARGET] = (data.groupby('candle_begin_time')[TARGET].rank(method='min', ascending=True)-1) / data['count']

In [27]:
for fea in tqdm(list(USE_COLS)):
    # 对每个特征按 'candle_begin_time' 分组，计算排名和归一化
    data[fea] = (data.groupby('candle_begin_time')[fea].rank(method='min', ascending=True)-1) / data['count']

100%|██████████| 73/73 [02:20<00:00,  1.92s/it]


In [28]:
data

,candle_begin_time,symbol,is_spot,close,f_BiasV1_12,f_BiasV1_168,f_BiasV1_24,f_BiasV1_360,f_BiasV1_72,f_BiasV1_720,...,f_ZfStd_72,f_ZfStd_720,f_动量因子1_12,f_动量因子1_168,f_动量因子1_24,f_动量因子1_360,f_动量因子1_72,f_动量因子1_720,Return_24,count
0,2021-01-01 00:00:00,AAVE-USDT,0,88.285004,0.333333,0.884615,0.602564,0.628205,0.833333,0.653846,...,0.512821,0.756410,0.474359,0.948718,0.769231,0.782051,0.935897,0.782051,0.717949,78
1,2021-01-01 00:00:00,ADA-USDT,0,0.180800,0.102564,0.692308,0.461538,0.846154,0.423077,0.871795,...,0.769231,0.423077,0.192308,0.628205,0.166667,0.794872,0.166667,0.807692,0.256410,78
2,2021-01-01 00:00:00,ALGO-USDT,0,0.346000,0.910256,0.653846,0.884615,0.602564,0.589744,0.641026,...,0.615385,0.589744,0.935897,0.692308,0.948718,0.705128,0.628205,0.666667,0.948718,78
3,2021-01-01 00:00:00,ALPHA-USDT,0,0.187750,0.769231,0.730769,0.846154,0.076923,0.525641,0.115385,...,0.551282,0.846154,0.820513,0.756410,0.923077,0.320513,0.589744,0.166667,0.961538,78
4,2021-01-01 00:00:00,ATOM-USDT,0,6.415000,0.371795,0.897436,0.628205,0.961538,0.871795,0.948718,...,0.794872,0.153846,0.397436,0.910256,0.705128,0.948718,0.897436,0.935897,0.064103,78
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6693955,2025-02-21 22:00:00,ZETA-USDT,0,0.309600,0.947075,0.707521,0.075209,0.860724,0.181058,0.300836,...,0.384401,0.311978,0.615599,0.657382,0.551532,0.657382,0.559889,0.459610,0.147632,359
6693956,2025-02-21 22:00:00,ZIL-USDT,0,0.014400,0.557103,0.816156,0.607242,0.364903,0.593315,0.389972,...,0.097493,0.128134,0.573816,0.573816,0.559889,0.665738,0.523677,0.657382,0.415042,359
6693957,2025-02-21 22:00:00,ZK-USDT,0,0.103830,0.534819,0.576602,0.150418,0.757660,0.208914,0.334262,...,0.551532,0.479109,0.693593,0.495822,0.665738,0.417827,0.342618,0.551532,0.278552,359
6693958,2025-02-21 22:00:00,ZRO-USDT,0,2.893300,0.412256,0.284123,0.122563,0.662953,0.217270,0.532033,...,0.598886,0.493036,0.740947,0.431755,0.662953,0.718663,0.284123,0.710306,0.192201,359


In [8]:
fea_df.groupby('candle_begin_time').size().sort_index()

candle_begin_time
2021-01-01 00:00:00     79
2021-01-01 01:00:00     79
2021-01-01 02:00:00     79
2021-01-01 03:00:00     79
2021-01-01 04:00:00     79
                      ... 
2025-02-21 18:00:00    360
2025-02-21 19:00:00    360
2025-02-21 20:00:00    360
2025-02-21 21:00:00    360
2025-02-21 22:00:00    360
Length: 36310, dtype: int64

In [9]:
# 任务
# lgb 划分训练集 验证集，在验证集上早停
# 验证集分别选择 2021 2022 2023 2024
# 统计 验证集 的 rank_ic 早停轮次  其他metric

In [10]:
1

1

In [11]:
for fea in USE_COLS:
    if data[fea].dtypes == 'object':
        data[fea] = data[fea].astype('float')
        print(f'{fea}已转换!')

In [15]:
test = data[pd.to_datetime(data[TIME]) >= pd.to_datetime('2024-01-01')].reset_index(drop=True)  # 测试集
train = data[pd.to_datetime(data[TIME]) < pd.to_datetime('2024-01-01')].reset_index(drop=True)  # 训练集

In [16]:
X_train = train[USE_COLS]
y_train = train[TARGET]
X_val = test[USE_COLS]
y_val = test[TARGET]

In [17]:
t=time.time()
lgb_train = lgb.Dataset(X_train, label = y_train)
lgb_eval = lgb.Dataset(X_val, label = y_val)
params = {
    'boosting_type': 'gbdt',
    'objective': 'regression',  # 将目标设置为回归
    'metric': 'rmse',  # 使用L2误差作为评估指标，或者你可以选择 'mae' 或 'rmse'
    'num_leaves': 128,
    'max_depth': 20,
    'min_data_in_leaf': 50,
    'learning_rate': 0.01,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'lambda_l1': 1,
    'lambda_l2': 1,  # 正则化，越小正则化效果越强
    'min_gain_to_split': 0.2,
    'verbose': -1,  # 关闭日志输出
    # 'device': 'gpu',  # 如果需要使用 GPU 训练，取消注释
}

# 训练回归模型
lgb_clf = lgb.train(
    params,
    lgb_train,
    num_boost_round=10000,
    valid_sets=lgb_eval,
    callbacks=[
        lgb.early_stopping(stopping_rounds=200),  # 提前停止
        lgb.log_evaluation(period=50)  # 每 50 次迭代打印一次日志
    ]
)
print(time.time()-t)
preds = lgb_clf.predict(X_val)
test['preds']=preds
hourly_columns= test.groupby('candle_begin_time')[['preds', 'Return_24']].corr(method='spearman').unstack().iloc[:, 1]
hourly_columns.mean()

Training until validation scores don't improve for 200 rounds
[50]	valid_0's rmse: 0.288401
[100]	valid_0's rmse: 0.287915
[150]	valid_0's rmse: 0.287748
[200]	valid_0's rmse: 0.287696
[250]	valid_0's rmse: 0.287693
[300]	valid_0's rmse: 0.287706
[350]	valid_0's rmse: 0.287726
[400]	valid_0's rmse: 0.287742
Early stopping, best iteration is:
[242]	valid_0's rmse: 0.287692
935.6381084918976


np.float64(0.11286298759971358)

In [18]:
test = data[pd.to_datetime(data[TIME]) < pd.to_datetime('2022-01-01')].reset_index(drop=True)  # 测试集
train = data[pd.to_datetime(data[TIME]) >= pd.to_datetime('2022-01-01')].reset_index(drop=True)  # 训练集
X_train = train[USE_COLS]
y_train = train[TARGET]
X_val = test[USE_COLS]
y_val = test[TARGET]
t=time.time()
lgb_train = lgb.Dataset(X_train, label = y_train)
lgb_eval = lgb.Dataset(X_val, label = y_val)
params = {
    'boosting_type': 'gbdt',
    'objective': 'regression',  # 将目标设置为回归
    'metric': 'rmse',  # 使用L2误差作为评估指标，或者你可以选择 'mae' 或 'rmse'
    'num_leaves': 128,
    'max_depth': 20,
    'min_data_in_leaf': 50,
    'learning_rate': 0.01,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'lambda_l1': 1,
    'lambda_l2': 1,  # 正则化，越小正则化效果越强
    'min_gain_to_split': 0.2,
    'verbose': -1,  # 关闭日志输出
    # 'device': 'gpu',  # 如果需要使用 GPU 训练，取消注释
}

# 训练回归模型
lgb_clf = lgb.train(
    params,
    lgb_train,
    num_boost_round=10000,
    valid_sets=lgb_eval,
    callbacks=[
        lgb.early_stopping(stopping_rounds=200),  # 提前停止
        lgb.log_evaluation(period=50)  # 每 50 次迭代打印一次日志
    ]
)
print(time.time()-t)
preds = lgb_clf.predict(X_val)
test['preds']=preds
hourly_columns= test.groupby('candle_begin_time')[['preds', 'Return_24']].corr(method='spearman').unstack().iloc[:, 1]
hourly_columns.mean()

Training until validation scores don't improve for 200 rounds
[50]	valid_0's rmse: 0.289596
[100]	valid_0's rmse: 0.288842
[150]	valid_0's rmse: 0.288469
[200]	valid_0's rmse: 0.288286
[250]	valid_0's rmse: 0.288175
[300]	valid_0's rmse: 0.288106
[350]	valid_0's rmse: 0.288071
[400]	valid_0's rmse: 0.288046
[450]	valid_0's rmse: 0.288041
[500]	valid_0's rmse: 0.288033
[550]	valid_0's rmse: 0.288034
[600]	valid_0's rmse: 0.288032
[650]	valid_0's rmse: 0.288041
[700]	valid_0's rmse: 0.28805
Early stopping, best iteration is:
[533]	valid_0's rmse: 0.28803
1813.0671434402466


np.float64(0.14105865235711113)

In [12]:
# 转换 TIME 列为 datetime 类型（如果数据中的 TIME 列还不是 datetime 类型的话）
data[TIME] = pd.to_datetime(data[TIME])
# 选择2022年整年的数据作为测试集
test = data[(data[TIME] >= pd.to_datetime('2022-01-01')) & (data[TIME] < pd.to_datetime('2023-01-01'))].reset_index(drop=True)
# 其余的数据作为训练集
train = data[~((data[TIME] >= pd.to_datetime('2022-01-01')) & (data[TIME] < pd.to_datetime('2023-01-01')))].reset_index(drop=True)

In [13]:
X_train = train[USE_COLS]
y_train = train[TARGET]
X_val = test[USE_COLS]
y_val = test[TARGET]
t=time.time()
lgb_train = lgb.Dataset(X_train, label = y_train)
lgb_eval = lgb.Dataset(X_val, label = y_val)
params = {
    'boosting_type': 'gbdt',
    'objective': 'regression',  # 将目标设置为回归
    'metric': 'rmse',  # 使用L2误差作为评估指标，或者你可以选择 'mae' 或 'rmse'
    'num_leaves': 128,
    'max_depth': 20,
    'min_data_in_leaf': 50,
    'learning_rate': 0.01,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'lambda_l1': 1,
    'lambda_l2': 1,  # 正则化，越小正则化效果越强
    'min_gain_to_split': 0.2,
    'verbose': -1,  # 关闭日志输出
    # 'device': 'gpu',  # 如果需要使用 GPU 训练，取消注释
}

# 训练回归模型
lgb_clf = lgb.train(
    params,
    lgb_train,
    num_boost_round=10000,
    valid_sets=lgb_eval,
    callbacks=[
        lgb.early_stopping(stopping_rounds=200),  # 提前停止
        lgb.log_evaluation(period=50)  # 每 50 次迭代打印一次日志
    ]
)
print(time.time()-t)
preds = lgb_clf.predict(X_val)
test['preds']=preds
hourly_columns= test.groupby('candle_begin_time')[['preds', 'Return_24']].corr(method='spearman').unstack().iloc[:, 1]
hourly_columns.mean()

Training until validation scores don't improve for 200 rounds
[50]	valid_0's rmse: 0.289271
[100]	valid_0's rmse: 0.288633
[150]	valid_0's rmse: 0.28834
[200]	valid_0's rmse: 0.288196
[250]	valid_0's rmse: 0.288125
[300]	valid_0's rmse: 0.288081
[350]	valid_0's rmse: 0.288052
[400]	valid_0's rmse: 0.288046
[450]	valid_0's rmse: 0.288043
[500]	valid_0's rmse: 0.288043
[550]	valid_0's rmse: 0.288054
[600]	valid_0's rmse: 0.28806
[650]	valid_0's rmse: 0.288069
Early stopping, best iteration is:
[474]	valid_0's rmse: 0.28804
1664.647115945816


np.float64(0.127576876597046)

In [14]:
# 转换 TIME 列为 datetime 类型（如果数据中的 TIME 列还不是 datetime 类型的话）
data[TIME] = pd.to_datetime(data[TIME])
# 选择2022年整年的数据作为测试集
test = data[(data[TIME] >= pd.to_datetime('2023-01-01')) & (data[TIME] < pd.to_datetime('2024-01-01'))].reset_index(drop=True)
# 其余的数据作为训练集
train = data[~((data[TIME] >= pd.to_datetime('2023-01-01')) & (data[TIME] < pd.to_datetime('2024-01-01')))].reset_index(drop=True)
X_train = train[USE_COLS]
y_train = train[TARGET]
X_val = test[USE_COLS]
y_val = test[TARGET]
t=time.time()
lgb_train = lgb.Dataset(X_train, label = y_train)
lgb_eval = lgb.Dataset(X_val, label = y_val)
params = {
    'boosting_type': 'gbdt',
    'objective': 'regression',  # 将目标设置为回归
    'metric': 'rmse',  # 使用L2误差作为评估指标，或者你可以选择 'mae' 或 'rmse'
    'num_leaves': 128,
    'max_depth': 20,
    'min_data_in_leaf': 50,
    'learning_rate': 0.01,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'lambda_l1': 1,
    'lambda_l2': 1,  # 正则化，越小正则化效果越强
    'min_gain_to_split': 0.2,
    'verbose': -1,  # 关闭日志输出
    # 'device': 'gpu',  # 如果需要使用 GPU 训练，取消注释
}

# 训练回归模型
lgb_clf = lgb.train(
    params,
    lgb_train,
    num_boost_round=10000,
    valid_sets=lgb_eval,
    callbacks=[
        lgb.early_stopping(stopping_rounds=200),  # 提前停止
        lgb.log_evaluation(period=50)  # 每 50 次迭代打印一次日志
    ]
)
print(time.time()-t)
preds = lgb_clf.predict(X_val)
test['preds']=preds
hourly_columns= test.groupby('candle_begin_time')[['preds', 'Return_24']].corr(method='spearman').unstack().iloc[:, 1]
hourly_columns.mean()

Training until validation scores don't improve for 200 rounds
[50]	valid_0's rmse: 0.288943
[100]	valid_0's rmse: 0.288488
[150]	valid_0's rmse: 0.288349
[200]	valid_0's rmse: 0.288315
[250]	valid_0's rmse: 0.288321
[300]	valid_0's rmse: 0.288331
[350]	valid_0's rmse: 0.288346
[400]	valid_0's rmse: 0.288359
Early stopping, best iteration is:
[218]	valid_0's rmse: 0.288313
1107.5090799331665


np.float64(0.10593720560325336)